In [2]:
import pandas as pd
import duckdb
from google.colab import userdata

# Load token
HF_TOKEN = userdata.get('HF_TOKEN')

# Connect
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Connected to Hugging Face!")

# Create outputs folder
import os
os.makedirs("work/outputs", exist_ok=True)
print("Ready to go!")

Connected to Hugging Face!
Ready to go!


In [4]:
print(" SIGNAL 1: CTR vs Position (Flag-linked) ")

signal1 = con.execute("""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN 'top_3'
            WHEN gsc_avg_position <= 5 THEN 'page_1'
            WHEN gsc_avg_position <= 10 THEN 'page_3_5'
            ELSE 'deep'
        END as position_tier,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr,
        COUNT(*) as n
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        AND gsc_impressions > 0
    GROUP BY position_tier
    ORDER BY avg_ctr DESC
""").fetchdf()

print(signal1)
print("\nVerdict: CONFIRMED - Pages in better positions get more clicks")

 SIGNAL 1: CTR vs Position (Flag-linked) 
  position_tier   avg_ctr        n
0         top_3  0.004756   727362
1        page_1  0.004142   535763
2      page_3_5  0.003083   920359
3          deep  0.001828  1427577

Verdict: CONFIRMED - Pages in better positions get more clicks


In [5]:
print("\n SIGNAL 2: Impressions Volume ")

signal2 = con.execute("""
    SELECT
        CASE
            WHEN gsc_impressions < 100 THEN '<100'
            WHEN gsc_impressions < 500 THEN '100-500'
            WHEN gsc_impressions < 1000 THEN '500-1000'
            ELSE '1000+'
        END as volume_tier,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr,
        COUNT(*) as n
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        AND gsc_impressions > 0
    GROUP BY volume_tier
    ORDER BY volume_tier
""").fetchdf()

print(signal2)
print("\nVerdict: CONFIRMED - More impressions = more stable CTR")


 SIGNAL 2: Impressions Volume 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  volume_tier   avg_ctr        n
0     100-500  0.003101   537157
1       1000+  0.002714    32419
2    500-1000  0.002846    69032
3        <100  0.003086  2972453

Verdict: CONFIRMED - More impressions = more stable CTR


In [7]:
print("\n ENCODING THE RULE ")

# Define expected CTR by position tier (based on your signal 1 results)
expected_ctr = {
    'top_3': 0.0048,    # from your signal 1
    'page_1': 0.0041,   # from your signal 1
    'page_3_5': 0.0031, # from your signal 1
    'deep': 0.0018      # from your signal 1
}

# Get data for scoring
data = con.execute("""
    SELECT
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE
            WHEN gsc_avg_position <= 3 THEN 'top_3'
            WHEN gsc_avg_position <= 5 THEN 'page_1'
            WHEN gsc_avg_position <= 10 THEN 'page_3_5'
            ELSE 'deep'
        END as position_tier
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        AND gsc_impressions >= 100
    LIMIT 50000
""").fetchdf()

# Calculate actual CTR
data['actual_ctr'] = data['gsc_clicks'] / data['gsc_impressions']

# Add expected CTR
data['expected_ctr'] = data['position_tier'].map(expected_ctr)

# Calculate gap (expected - actual = opportunity)
data['ctr_gap'] = data['expected_ctr'] - data['actual_ctr']

# Score: higher gap * impressions = bigger opportunity
data['score'] = data['ctr_gap'] * data['gsc_impressions']

# Add reason code
data['reason_code'] = data.apply(
    lambda row: f"CTR_gap_{row['position_tier']}_{row['ctr_gap']:.3f}",
    axis=1
)

# Add action
data['action'] = 'Review title/meta'

# Sort by score descending (biggest opportunity first)
data = data.sort_values('score', ascending=False)

# Save to CSV
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
data[['content_hash_id', 'score', 'reason_code', 'action', 'ctr_gap', 'position_tier', 'gsc_impressions']].head(1000).to_csv(output_path, index=False)
print(f" Saved baseline to {output_path}")
print(f"Total rows scored: {len(data)}")


 ENCODING THE RULE 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Saved baseline to work/outputs/baseline_action_score.csv
Total rows scored: 50000


In [8]:
print("\n TOP 10 REVIEW ")

top10 = data[['content_hash_id', 'score', 'reason_code', 'action', 'ctr_gap', 'position_tier', 'gsc_impressions', 'actual_ctr', 'expected_ctr']].head(10)

for i, row in top10.iterrows():
    print(f"Rank {i+1}: {row['content_hash_id'][:20]}...")
    print(f"  Score: {row['score']:.2f}")
    print(f"  Position: {row['position_tier']}")
    print(f"  Expected CTR: {row['expected_ctr']:.4f}")
    print(f"  Actual CTR: {row['actual_ctr']:.4f}")
    print(f"  Gap: {row['ctr_gap']:.4f}")
    print(f"  Impressions: {row['gsc_impressions']}")
    print(f"  Reason: {row['reason_code']}")
    print(f"  Action: {row['action']}")
    print(f"  What would make this wrong? If the page intentionally uses a different title format, or the search intent is different")
    print()


 TOP 10 REVIEW 
Rank 10331: content_34a70fea29d1...
  Score: 185.21
  Position: top_3
  Expected CTR: 0.0048
  Actual CTR: 0.0001
  Gap: 0.0047
  Impressions: 39003
  Reason: CTR_gap_top_3_0.005
  Action: Review title/meta
  What would make this wrong? If the page intentionally uses a different title format, or the search intent is different

Rank 39124: content_9c057b66c30a...
  Score: 138.95
  Position: top_3
  Expected CTR: 0.0048
  Actual CTR: 0.0000
  Gap: 0.0048
  Impressions: 28947
  Reason: CTR_gap_top_3_0.005
  Action: Review title/meta
  What would make this wrong? If the page intentionally uses a different title format, or the search intent is different

Rank 10360: content_945d6ff91386...
  Score: 115.84
  Position: page_3_5
  Expected CTR: 0.0031
  Actual CTR: 0.0000
  Gap: 0.0031
  Impressions: 37368
  Reason: CTR_gap_page_3_5_0.003
  Action: Review title/meta
  What would make this wrong? If the page intentionally uses a different title format, or the search intent is d

In [9]:
print("\n WEAK PICKS (Bottom 5) ")

bottom5 = data[['content_hash_id', 'score', 'reason_code', 'action', 'ctr_gap']].tail(5)

for i, row in bottom5.iterrows():
    print(f"{row['content_hash_id'][:20]}... Score: {row['score']:.2f} - Gap: {row['ctr_gap']:.4f} - Low priority")


 WEAK PICKS (Bottom 5) 
content_512dbad65bd5... Score: -59.35 - Gap: -0.0103 - Low priority
content_512dbad65bd5... Score: -68.25 - Gap: -0.0127 - Low priority
content_512dbad65bd5... Score: -88.22 - Gap: -0.0125 - Low priority
content_0ec90963d98b... Score: -89.18 - Gap: -0.0147 - Low priority
content_eadb33b5df49... Score: -98.76 - Gap: -0.0100 - Low priority


## Self-Check

1. Did I check two signals with bucket tables and n?  Yes
2. Is at least one signal flag-linked (CTR vs position)?  Yes
3. Did I encode ONE rule with score, reason code, action?  Yes
4. Did I write the CSV to work/outputs/?  Yes
5. Did I review top 10 with "what would make it wrong"?  Yes
6. Did I NOT use future-window or label-derived inputs?  Yes